# M05 · Offline Metrics — Toy Example, Step by Tiny Step

**Companion to lesson M05.** Turn model scores into a **confusion matrix**, read off **precision / recall / F1**, sweep the **threshold**, and meet the **imbalance trap** where accuracy lies.

## Step 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)

def log(label, value):
    print(f"[{label}] {value}")

log("setup", "tools ready — seed fixed to 0")

## Step 1 · Confusion matrix and precision/recall/F1

At a chosen threshold, every prediction is a TP, FP, FN, or TN. Precision = of what I flagged, how many were right; recall = of the true positives, how many I caught.

In [ ]:
TP, FP, FN, TN = 8, 2, 4, 86                    # from a threshold on synthetic pCTR scores
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
log("precision = TP/(TP+FP)", f"{TP}/{TP+FP} = {precision:.2f}")
log("recall = TP/(TP+FN)", f"{TP}/{TP+FN} = {recall:.2f}")
log("F1", round(f1, 3))
assert abs(precision - 0.80) < 1e-9 and abs(recall - 2/3) < 1e-9

cm = np.array([[TN, FP], [FN, TP]])
plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xticks([0,1], ["pred -","pred +"]); plt.yticks([0,1], ["true -","true +"])
plt.title("confusion matrix"); plt.show()

▶ What you'll see: precision 0.80, recall 0.67, F1 ≈ 0.73.

## Step 2 · Threshold sweep — recall falls as the threshold rises

Raise the threshold and you predict positive less often: fewer false alarms (precision up) but more misses (recall down). We sweep and plot both.

In [ ]:
y = np.array([1]*20 + [0]*80)                   # 20 positives, 80 negatives
scores = np.clip(np.where(y == 1, np.random.normal(0.65, 0.15, 100),
                          np.random.normal(0.35, 0.15, 100)), 0, 1)
ths = np.linspace(0.1, 0.9, 9); precs, recs = [], []
for t in ths:
    p = (scores >= t).astype(int)
    tp = np.sum((y==1)&(p==1)); fp = np.sum((y==0)&(p==1)); fn = np.sum((y==1)&(p==0))
    precs.append(tp/(tp+fp) if tp+fp else 1.0); recs.append(tp/(tp+fn) if tp+fn else 0.0)
log("recall by threshold", [round(r,2) for r in recs])
assert all(recs[i+1] <= recs[i] + 1e-9 for i in range(len(recs)-1))   # recall never rises with threshold

plt.plot(ths, precs, "-o", label="precision"); plt.plot(ths, recs, "-o", label="recall")
plt.title("precision & recall vs threshold"); plt.xlabel("threshold"); plt.legend(); plt.show()

▶ What you'll see: recall sliding down as the threshold rises, precision generally rising.

## Step 3 · The imbalance trap (break case)

On a 1%-click dataset, predicting **no clicks** is ~99% accurate and catches **zero** clicks.

In [ ]:
y_imb = np.array([1] + [0]*99)                  # 1% positive
pred_none = np.zeros(100, int)
log("accuracy (predict no clicks)", np.mean(pred_none == y_imb))
log("recall (predict no clicks)", 0.0)
assert np.mean(pred_none == y_imb) == 0.99
print("Lesson: on imbalanced data use precision/recall/F1/AUC, never accuracy.")

▶ What you'll see: 0.99 accuracy next to 0.0 recall.

## Recap

- Precision/recall/F1 are fractions of the four confusion counts.
- The threshold trades precision for recall — sweep it, don't guess.
- On imbalanced data, **accuracy lies**; judge by recall/precision/F1/AUC.